In [1]:
import warnings

warnings.filterwarnings("ignore")

import os, gc
import pandas as pd
import numpy as np
from tqdm import tqdm

possible_bases = [
    "/kaggle/input/uw-madison-gi-tract-image-segmentation",
    "./input/uw-madison-gi-tract-image-segmentation",
    "./data/uw-madison-gi-tract-image-segmentation",
    "./uw-madison-gi-tract-image-segmentation",
]
BASE_PATH = next((p for p in possible_bases if os.path.isdir(p)), None)
if BASE_PATH is None:
    raise FileNotFoundError("Could not locate the competition data directory.")

test_path = os.path.join(BASE_PATH, "test.csv")
if os.path.exists(test_path):
    df = pd.read_csv(test_path)
else:
    train_path = os.path.join(BASE_PATH, "train.csv")
    df = pd.read_csv(train_path, usecols=["id", "class"])
    df["predicted"] = ""

train_mask_path = os.path.join(BASE_PATH, "train.csv")
train_df = pd.read_csv(train_mask_path, usecols=["class", "segmentation"])


def rle_decode(rle_str, shape=(256, 256)):
    """Decode a run‑length encoded string to a binary mask using column‑major order."""
    if pd.isna(rle_str) or rle_str == "":
        return np.zeros(shape, dtype=np.uint8)
    s = list(map(int, rle_str.split()))
    starts = s[0::2]
    lengths = s[1::2]
    flat = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for start, length in zip(starts, lengths):
        flat[start - 1 : start - 1 + length] = 1
    # Use Fortran (column‑major) order to match the competition specification
    return flat.reshape(shape, order="F")




In [2]:
sum_masks = {
    "large_bowel": np.zeros((256, 256), dtype=np.uint16),
    "small_bowel": np.zeros((256, 256), dtype=np.uint16),
    "stomach": np.zeros((256, 256), dtype=np.uint16),
}
class_counts = {"large_bowel": 0, "small_bowel": 0, "stomach": 0}

for _, row in tqdm(
    train_df.iterrows(), total=train_df.shape[0], desc="Aggregating masks"
):
    cls = row["class"]
    if cls in sum_masks:
        mask = rle_decode(row["segmentation"])
        sum_masks[cls] += mask.astype(np.uint16)
        class_counts[cls] += 1

class_masks = {}
for cls, agg_mask in sum_masks.items():
    class_masks[cls] = (agg_mask >= 1).astype(np.uint8)




Aggregating masks: 100%|██████████| 95088/95088 [00:04<00:00, 19036.23it/s]


In [3]:
def safe_split(id_str, idx):
    parts = id_str.split("_")
    return parts[idx] if idx < len(parts) else ""


df["case"] = df["id"].apply(
    lambda x: int(safe_split(x, 0).replace("case", "")) if safe_split(x, 0) else -1
)
df["day"] = df["id"].apply(
    lambda x: int(safe_split(x, 1).replace("day", "")) if safe_split(x, 1) else -1
)
df["slice"] = df["id"].apply(lambda x: safe_split(x, 3))

df["width"] = 256
df["height"] = 256
gc.collect()




0

In [4]:
ids, classes, rles = [], [], []


def rle_encode(img):
    """Encode binary mask img (2D numpy array of 0/1) to run‑length string using column‑major order."""
    # Flatten using Fortran (column‑major) order to follow the competition specification
    pixels = img.flatten(order="F")
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)


for _, row in tqdm(df.iterrows(), total=df.shape[0], desc="Creating submission"):
    height, width = int(row["height"]), int(row["width"])
    cls = row["class"]
    mask = class_masks.get(cls, np.zeros((height, width), dtype=np.uint8))
    rle = rle_encode(mask)
    ids.append(row["id"])
    classes.append(cls)
    rles.append(rle)




Creating submission: 100%|██████████| 20400/20400 [00:03<00:00, 5670.54it/s]


In [5]:
submission = pd.DataFrame({"id": ids, "class": classes, "predicted": rles})
submission = submission.sort_values(["id", "class"]).reset_index(drop=True)

output_path = os.path.join(os.getcwd(), "submission.csv")
submission.to_csv(output_path, index=False)

print(f"Submission file '{output_path}' written with {submission.shape[0]} rows.")

Submission file '/kaggle/working/submission.csv' written with 20400 rows.
